In [ ]:
import os
from collections import defaultdict
from importlib import resources as impresources

import matplotlib as mpl
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from tqdm.auto import tqdm, trange

import mcfacts.vis.LISA as li

from mcfacts.fiducial_plots import make_gen_masks
from mcfacts.inputs.settings_manager import AGNDisk, SettingsManager
from mcfacts.modules.accretion import ProgradeBlackHoleAccretion, BinaryBlackHoleAccretion
from mcfacts.modules.damping import ProgradeBlackHoleDamping, BinaryBlackHoleDamping
from mcfacts.modules.disk_capture import EvolveRetrogradeBlackHoles, RecaptureBinaryBlackHoles, \
    CaptureNSCProgradeBlackHoles
from mcfacts.modules.dynamics import SingleBlackHoleDynamics, BinaryBlackHoleDynamics, BinaryBlackHoleEccDynamics, \
    BinaryBlackHoleSpheroidDynamics, BinaryBlackHoleIonization
from mcfacts.modules.formation import BinaryBlackHoleFormation
from mcfacts.modules.gas_hardening import BinaryBlackHoleGasHardening
from mcfacts.modules.gw import InnerBlackHoleDynamics, BinaryBlackHoleEvolveGW
from mcfacts.modules.merge import ProcessBinaryBlackHoleMergers, ProcessEMRIMergers
from mcfacts.modules.migration import ProgradeBlackHoleMigration, BinaryBlackHoleMigration
from mcfacts.objects.actors import InitialBlackHoleReclassification, SingleBlackHoleRealityCheck, \
    BinaryBlackHoleRealityCheck, InnerDiskFilter, FlipRetroProFilter
from mcfacts.objects.agn_object_array import FilingCabinet, AGNBinaryBlackHoleArray
from mcfacts.objects.galaxy import Galaxy
from mcfacts.objects.populators import SingleBlackHolePopulator
from mcfacts.objects.snapshot import TxtSnapshotHandler, IniSnapshotHandler
from mcfacts.objects.timeline import SimulationTimeline, TimelineActor
from mcfacts.vis import data, plotting, styles

plt.style.use("mcfacts.vis.mcfacts_figures")

In [ ]:
# Custom timeline actor added to the simulation timeline to keep track of values not currently tracked by the canonical simulation.

class PopulationStateRecorder(TimelineActor):
    def __init__(self, history, galaxy_id, settings=None):
        super().__init__("Population State Recorder", settings)
        self.history = history
        self.galaxy_id = galaxy_id

    def __deepcopy__(self, memo):
        return self

    def perform(self, timestep, timestep_length, time_passed, filing_cabinet, agn_disk, random_generator):
        self.history.append({
            "galaxy_id": self.galaxy_id,
            "timestep": timestep,
            "filing_cabinet": filing_cabinet,
        })

# class PopulationStateRecorder(TimelineActor):
#     def __init__(self, history, galaxy_id, settings=None):
#         super().__init__("Population State Recorder", settings)
#         self.history = history
#         self.galaxy_id = galaxy_id
#
#     def __deepcopy__(self, memo):
#         # Return self on deepcopy, maintaining reference to history throughout timeline executions.
#         return self
#
#     def perform(self, timestep, timestep_length, time_passed, filing_cabinet, agn_disk, random_generator):
#         sm = self.settings
#
#         object_array_names = [sm.bh_prograde_array_name, sm.bh_retrograde_array_name, sm.bh_inner_disk_array_name, sm.bh_inner_gw_array_name]
#         object_mass_arrays = []
#
#         for array_name in object_array_names:
#             if array_name not in filing_cabinet:
#                 continue
#
#             object_array = filing_cabinet.get_array(array_name)
#
#             if len(object_array) == 0:
#                 continue
#
#             object_mass_arrays.append(object_array.mass)
#
#         masses = np.concatenate(object_mass_arrays)
#
#         n_binaries, mean_sep, total_binary_mass = 0, np.nan, 0.0
#         if sm.bbh_array_name in filing_cabinet:
#             binaries_array = filing_cabinet.get_array(sm.bbh_array_name, AGNBinaryBlackHoleArray)
#
#             n_binaries = len(binaries_array)
#             mean_sep = float(np.mean(binaries_array.bin_sep)) if n_binaries > 0 else np.nan
#             total_binary_mass = float(np.sum(binaries_array.mass) + np.sum(binaries_array.mass_2)) if n_binaries > 0 else 0.0
#
#         total_n_merged = len(filing_cabinet.get_array(sm.bbh_merged_array_name)) if sm.bbh_merged_array_name in filing_cabinet else 0
#
#         self.history.append({
#             "galaxy_id": self.galaxy_id,
#             "timestep": timestep,
#             "time_passed": time_passed,
#             "n_singles": len(masses),
#             "n_binaries": n_binaries,
#             "total_n_merged": total_n_merged,
#             "mean_mass": float(np.mean(masses)),
#             "median_mass": float(np.median(masses)),
#             "max_mass": float(np.max(masses)),
#             "total_mass": float(np.sum(masses)) + total_binary_mass,
#             "bin_sep_mean": mean_sep,
#         })

In [ ]:
# Simulation Settings
RUNS_DIR = "./runs"

GALAXY_NUM = 100
MIG_TORQUE_PRESCRIPTION = "jimenez_masset"
FRACTION_BIN_RETRO = 0.5

# No stalling, fixed stalling models, and quasi-dynamical stalling
STALLING_VALUES = [0.0, 0.25, 1.0, 5.0, -1.0]

# Baruteau, Stahler, and Semi-Analytical gas hardening
GAS_HARDENING_PRESCRIPTIONS = ["baruteau", "stahler"]

# m^-2 Initial mass function and guassian at 35
INITIAL_MASS_FUNCTIONS = ["default", "gaussian"]

CIRC_HARDENING_MU = [0.2, 0.5, 0.7, 0.9]
# CIRC_HARDENING_SIGMA = [0.025, 0.1] # Run 0.1 for stalling at 0.5
ECC_HARDENING_MU = [0.1, 0.2]
# ECC_HARDENING_SIGMA = [0.02, 0.1] # Run 0.1 for stalling 0.5

# Finny and sigerton, ecc energy exchange at 20% in clusters.

GAUSS_MU = 35 # Pull from GWTC-5
GAUSS_SIGMA = 2.3

CIRC_HARDENING_TAGS = {
    0.2: "0p2",
    0.5: "0p5",
    0.7: "0p7",
    0.9: "0p9",
}

ECC_HARDENING_TAGS = {
    0.1: "0p1",
    0.2: "0p2",
}

STALLING_TAGS = {
    0.0: "none",
    0.25: "0p25",
    0.5: "0p5",
    1.0: "1",
    2.0: "2",
    5.0: "5",
    -1.0: "variable"
}

STALLING_LABELS = {
    0.0: "no stalling",
    0.25: r"0.25 $r_g$",
    0.5: r"0.5 $r_g$",
    1.0: r"1 $r_g$",
    2.0: r"2 $r_g$",
    5.0: r"5 $r_g$",
    -1.0: r"variable ($\mathcal{M}=1$)"
}

FIG_DIR = "./figures"

FIGSIZE = "apj_col"
W, H = plotting.set_size(FIGSIZE)

FORMATS = ("png", "pdf", "svg")

PAIR_COLORS = {"nostall": "tab:blue", "stall": "tab:red"}

In [ ]:
# Build out the permutations of configs for each of the values we're studying.

def build_configs(galaxy_num):
    configs = {}

    def add_config(config_tag, overrides):
        static_settings = dict(galaxy_num=galaxy_num, torque_prescription=MIG_TORQUE_PRESCRIPTION, fraction_bin_retro=FRACTION_BIN_RETRO)
        combined_settings = static_settings | overrides

        configs[config_tag] = SettingsManager(settings_overrides=combined_settings)

    for imf in INITIAL_MASS_FUNCTIONS:
        for circ_hardening_mu in CIRC_HARDENING_MU:
            if circ_hardening_mu == 0.9:
                for ecc_hardening_mu in ECC_HARDENING_MU:
                    add_config(f"analytical_imf-{imf}_circ-{CIRC_HARDENING_TAGS[circ_hardening_mu]}_ecc-{ECC_HARDENING_TAGS[ecc_hardening_mu]}", dict(
                        gas_hardening_prescription="analytical",
                        mean_harden_energy_delta=circ_hardening_mu,
                        delta_energy_strong_mu=ecc_hardening_mu,
                        nsc_imf_bh_method=imf,
                        mass_pile_up=GAUSS_MU,
                        nsc_imf_bh_mode=GAUSS_SIGMA,
                        flag_phenom_turb=False
                    ))
            else:
                add_config(f"analytical_imf-{imf}_circ-{CIRC_HARDENING_TAGS[circ_hardening_mu]}", dict(
                    gas_hardening_prescription="analytical",
                    mean_harden_energy_delta=circ_hardening_mu,
                    nsc_imf_bh_method=imf,
                    mass_pile_up=GAUSS_MU,
                    nsc_imf_bh_mode=GAUSS_SIGMA,
                    flag_phenom_turb=False
                ))


        for gas_hardening_prescription in GAS_HARDENING_PRESCRIPTIONS:
            for stalling_separation in STALLING_VALUES:
                if stalling_separation == 0 or stalling_separation == 1:
                    for circ_hardening_mu in CIRC_HARDENING_MU:
                        if circ_hardening_mu == 0.9:
                            for ecc_hardening_mu in ECC_HARDENING_MU:
                                add_config(f"{gas_hardening_prescription}_stall-{STALLING_TAGS[stalling_separation]}_imf-{imf}_circ-{CIRC_HARDENING_TAGS[circ_hardening_mu]}_ecc-{ECC_HARDENING_TAGS[ecc_hardening_mu]}", dict(
                                    gas_hardening_prescription=gas_hardening_prescription,
                                    stalling_separation=stalling_separation,
                                    mean_harden_energy_delta=0.9,
                                    delta_energy_strong_mu=ecc_hardening_mu,
                                    flag_phenom_turb=False,
                                    nsc_imf_bh_method=imf,
                                    mass_pile_up=GAUSS_MU,
                                    nsc_imf_bh_mode=GAUSS_SIGMA
                                ))
                        else:
                            add_config(f"{gas_hardening_prescription}_stall-{STALLING_TAGS[stalling_separation]}_imf-{imf}_circ-{CIRC_HARDENING_TAGS[circ_hardening_mu]}", dict(
                                gas_hardening_prescription=gas_hardening_prescription,
                                stalling_separation=stalling_separation,
                                mean_harden_energy_delta=circ_hardening_mu,
                                flag_phenom_turb=False,
                                nsc_imf_bh_method=imf,
                                mass_pile_up=GAUSS_MU,
                                nsc_imf_bh_mode=GAUSS_SIGMA
                            ))

                else:
                    add_config(f"{gas_hardening_prescription}_stall-{STALLING_TAGS[stalling_separation]}_imf-{imf}", dict(
                        gas_hardening_prescription=gas_hardening_prescription,
                        stalling_separation=stalling_separation,
                        flag_phenom_turb=False,
                        nsc_imf_bh_method=imf,
                        mass_pile_up=GAUSS_MU,
                        nsc_imf_bh_mode=GAUSS_SIGMA
                    ))



    return configs

In [ ]:
CONFIGS = build_configs(galaxy_num=GALAXY_NUM)

for i in CONFIGS.keys():
    print(i)

print(" ")
print(f"{len(CONFIGS)} configurations loaded, {(len(CONFIGS) * 18) / 60}m estimated run time")

In [ ]:
# *** Main simulation script ***
AGN_DISK = AGNDisk(settings=list(CONFIGS.values())[0])

RESULTS = {}

pbar = tqdm(CONFIGS.items(), position=0)

for name, settings in pbar:
    pbar.set_description(name)

    out_dir = os.path.join(RUNS_DIR, name)

    population_cabinet = FilingCabinet()
    population_cabinet.ignore_consistency_check("blackholes_merged")
    population_cabinet.ignore_consistency_check("blackholes_lvk")

    history = []

    # Start galaxy loop
    for galaxy_id in range(settings.galaxy_num):
        galaxy_seed = settings.seed - galaxy_id
        galaxy = Galaxy(seed=galaxy_seed, runs_folder=out_dir, galaxy_id=str(galaxy_id), settings=settings)

        galaxy.populate([SingleBlackHolePopulator()], AGN_DISK)

        pre_timeline = SimulationTimeline("Reclassification", timesteps=1, timestep_length=0)
        pre_timeline.add_timeline_actor(InitialBlackHoleReclassification())
        pre_timeline.add_timeline_actor(SingleBlackHoleRealityCheck())
        galaxy.run(pre_timeline, AGN_DISK)

        active_timeline = SimulationTimeline(
            "Active Timeline",
            timesteps=settings.active_timestep_num,
            timestep_length=galaxy.settings.active_timestep_duration_yr,
        )

        active_timeline.add_timeline_actor(SingleBlackHoleRealityCheck())

        population_recorder_actor = PopulationStateRecorder(history, galaxy_id, settings)
        active_timeline.add_timeline_actor(population_recorder_actor)

        prograde_array = galaxy.settings.bh_prograde_array_name
        innerdisk_array = galaxy.settings.bh_inner_disk_array_name
        inner_gw_only_array = galaxy.settings.bh_inner_gw_array_name

        active_timeline.add_timeline_actors([
            ProgradeBlackHoleMigration(target_array=innerdisk_array),
            ProgradeBlackHoleMigration(target_array=prograde_array),
            SingleBlackHoleRealityCheck(),

            ProgradeBlackHoleAccretion(target_array=innerdisk_array),
            ProgradeBlackHoleAccretion(target_array=prograde_array),
            ProgradeBlackHoleDamping(target_array=innerdisk_array),
            ProgradeBlackHoleDamping(target_array=prograde_array),

            EvolveRetrogradeBlackHoles(),
            SingleBlackHoleRealityCheck(),

            InnerBlackHoleDynamics(target_array=innerdisk_array),
            InnerBlackHoleDynamics(target_array=inner_gw_only_array),
            SingleBlackHoleDynamics(target_array=innerdisk_array),
            SingleBlackHoleDynamics(target_array=prograde_array),
        ])

        active_timeline.add_timeline_actors([
            BinaryBlackHoleDamping(),

            BinaryBlackHoleDynamics(reality_merge_checks=False),
            ProcessBinaryBlackHoleMergers(),

            BinaryBlackHoleEccDynamics(reality_merge_checks=False),
            ProcessBinaryBlackHoleMergers(),

            BinaryBlackHoleGasHardening(reality_merge_checks=False),
            ProcessBinaryBlackHoleMergers(),

            BinaryBlackHoleAccretion(reality_merge_checks=False),
            ProcessBinaryBlackHoleMergers(),

            BinaryBlackHoleSpheroidDynamics(reality_merge_checks=False),
            ProcessBinaryBlackHoleMergers(),

            RecaptureBinaryBlackHoles(),
            BinaryBlackHoleMigration(),
            BinaryBlackHoleRealityCheck(),

            BinaryBlackHoleEvolveGW(),

            BinaryBlackHoleIonization(),
            ProcessBinaryBlackHoleMergers(),

            BinaryBlackHoleFormation()
        ])

        active_timeline.add_timeline_actor(CaptureNSCProgradeBlackHoles())
        active_timeline.add_timeline_actor(InnerDiskFilter())
        active_timeline.add_timeline_actor(FlipRetroProFilter())
        active_timeline.add_timeline_actor(ProcessEMRIMergers())

        galaxy.run(active_timeline, AGN_DISK)

        history.append({
            "galaxy_id": galaxy_id,
            "timestep": settings.active_timestep_num,
            "filing_cabinet": galaxy.filing_cabinet,
        })

        population_arrays = {
            "blackholes_merged": [settings.bbh_merged_array_name],
            "blackholes_lvk": [settings.bbh_gw_array_name],
            "blackholes_ejected": [settings.bh_ejected_array_name],
            "blackholes_emri": [settings.bh_inner_disk_array_name, settings.bh_inner_gw_array_name, settings.emri_array_name],
        }

        for key, value in population_arrays.items():
            for object_array_name in value:
                if object_array_name not in galaxy.filing_cabinet:
                    continue

                object_array = galaxy.filing_cabinet.get_array(object_array_name)
                object_array.galaxy_id = np.full(len(object_array.unique_id), galaxy_id)

                population_cabinet.create_or_append_array(key, object_array)
    # End galaxy loop

    # Create the output directory for this run
    os.makedirs(out_dir, exist_ok=True)

    # Save run files
    TxtSnapshotHandler(settings=settings).save_cabinet(out_dir, "population", population_cabinet)
    IniSnapshotHandler(settings=settings).save_settings(out_dir, "settings", settings)
    pd.DataFrame(history).to_csv(os.path.join(out_dir, "history.csv"), index=False)

    RESULTS[name] = {
        "out_dir": out_dir,
        "settings": settings,
        "cabinet": population_cabinet,
        "history": pd.DataFrame(history),
    }

In [ ]:
def object_array_to_df(result, key):
    """One population array -> DataFrame, with galaxy provenance attached."""
    population_cabinet = result["cabinet"]

    if key not in population_cabinet.agn_objects:
        return pd.DataFrame()

    object_array = population_cabinet.agn_objects[key]
    df = pd.DataFrame({k: np.asarray(v) for k, v in object_array.get_super_dict().items()})

    if not df.empty and {"mass", "mass_2"}.issubset(df.columns):
        m1 = df["mass"].to_numpy(dtype=float)
        m2 = df["mass_2"].to_numpy(dtype=float)
        df["mass_ratio"] = np.minimum(m1, m2) / np.maximum(m1, m2)

    return df

In [ ]:
summary_rows = []
for name, result in RESULTS.items():
    merged = object_array_to_df(result, "blackholes_merged")

    galaxy_num = result["settings"].galaxy_num

    per_galaxy = merged.galaxy_id.value_counts()

    summary_rows.append({
        "name": name,
        "n_galaxy": galaxy_num,
        "n_merged": len(merged),
        "n_lvk": len(object_array_to_df(result, "blackholes_lvk")),
        "n_emri": len(object_array_to_df(result, "blackholes_emri")),
        "n_ejected": len(object_array_to_df(result, "blackholes_ejected")),
        "mergers_per_galaxy": per_galaxy.mean(),
        "mergers_per_galaxy_std": per_galaxy.std(),
    })

SUMMARY = pd.DataFrame(summary_rows)

SUMMARY.set_index('name', inplace=True)

os.makedirs(FIG_DIR, exist_ok=True)
SUMMARY.to_csv(os.path.join(FIG_DIR, "summary_table.csv"), index=False)

SUMMARY

In [ ]:
def save_figure(fig, name, fig_dir=FIG_DIR):
    os.makedirs(fig_dir, exist_ok=True)
    paths = []

    for ext in FORMATS:
        path = os.path.join(fig_dir, f"{name}.{ext}")
        fig.savefig(path, format=ext, bbox_inches="tight",
                    dpi=300 if ext == "png" else None)
        paths.append(path)

    return paths


def select_with_settings(results, **criteria):
    out = []

    for run_name, result in results.items():
        result_settings = result["settings"].settings_finals

        if all(result_settings.get(key) == value for key, value in criteria.items()):
            out.append(run_name)

    return out

In [ ]:
select_with_settings(
            RESULTS,
            gas_hardening_prescription="analytical",
            nsc_imf_bh_method="default",
            mean_harden_energy_delta=0.9,
            delta_energy_strong_mu=0.1
        )

In [ ]:
from matplotlib.legend_handler import HandlerErrorbar
from matplotlib.container import ErrorbarContainer

# Mergers vs Stalling
fig, (ax, ax_var) = plt.subplots(1, 2, figsize=(W * 1.45, H * 1.2), sharey=True, gridspec_kw={"width_ratios": [4, 1], "wspace": 0.05})

for presc, color in zip(["baruteau", "stahler"], ["tab:blue", "tab:red"]):
    for imf in INITIAL_MASS_FUNCTIONS[:1]:
        rows = {
            f"{STALLING_TAGS[s]}": select_with_settings(
                RESULTS,
                stalling_separation=s,
                gas_hardening_prescription=presc,
                nsc_imf_bh_method=imf,
                mean_harden_energy_delta=0.9,
                delta_energy_strong_mu=0.1

            )
            for s in STALLING_VALUES
        }

        y = []
        e = []

        for stalling_value in STALLING_VALUES[:4]:
            for run_name in rows[STALLING_TAGS[stalling_value]]:
                sum_data = SUMMARY.loc[run_name]
                y.append(sum_data.mergers_per_galaxy)
                e.append(sum_data.mergers_per_galaxy_std / np.sqrt(sum_data.n_galaxy))
                break

        ax.errorbar(STALLING_VALUES[:4], y, yerr=e, ls="-", marker="o", ms=4, capsize=2, lw=1, color=color)

        v = select_with_settings(
            RESULTS,
            stalling_separation=-1,
            gas_hardening_prescription=presc,
            nsc_imf_bh_method=imf,
            mean_harden_energy_delta=0.9,
            delta_energy_strong_mu=0.1
        )

        v_data = SUMMARY.loc[v]
        vy = v_data.mergers_per_galaxy
        ve = v_data.mergers_per_galaxy_std / np.sqrt(sum_data.n_galaxy)

        ax_var.errorbar([0], [vy], yerr=[ve], ls="none", marker="o", ms=4, capsize=2, color=color)

w = select_with_settings(
    RESULTS,
    stalling_separation=0,
    gas_hardening_prescription="analytical",
    nsc_imf_bh_method="default",
    mean_harden_energy_delta=0.9,
    delta_energy_strong_mu=0.1
)

w_data = SUMMARY.loc[w]
wy = w_data.mergers_per_galaxy
we = w_data.mergers_per_galaxy_std / np.sqrt(sum_data.n_galaxy)

ax_var.errorbar([1], [wy], yerr=[we], ls="none", marker="o", ms=4, capsize=2, color="tab:purple")

ax.errorbar([], [], yerr=[], ls="none", marker="o", ms=4, capsize=2, color="tab:blue", label="baruteau, default IMF")
ax.errorbar([], [], yerr=[], ls="none", marker="o", ms=4, capsize=2, color="tab:red", label="stahler, default IMF")
ax.errorbar([], [], yerr=[], ls="none", marker="o", ms=4, capsize=2, color="tab:purple", label="analytical, default IMF")


ax.set_xlabel(r"stalling separation [$r_g$]")
ax.set_ylabel("mergers per galaxy")

#ax.axvline(2.0, color="grey", ls=":", lw=1)
#ax.text(2.05, ax.get_ylim()[1] * 0.96, "GW tracking\nthreshold", fontsize=5, color="grey", va="top")

ax.legend(fontsize=5, frameon=False, handler_map={ErrorbarContainer: HandlerErrorbar(xerr_size=0, yerr_size=0)}, numpoints=1)

ax_var.set_xticks([0, 1], [r"variable" "\n" r"($\mathcal{M}=1$)", r"variable" "\n" r"(analytical)"], fontsize=5)
ax_var.tick_params(axis="y", left=False)
ax_var.set_xlim(-0.5, 1.5)

fig.suptitle("Number of Mergers vs Stalling Separation", fontsize=7)

save_figure(fig, "mergers_vs_stalling", FIG_DIR)

In [ ]:
def stall_color(stall, stalling_values):
    if stall == -1.0:
        return "black"

    exclude_var = dict([(s, i) for i, s in enumerate(stalling_values) if s >= 0])
    index = exclude_var[stall]

    return plt.cm.tab10(index)

fig = plt.figure(figsize=(W, H * 1.2))
ax = fig.add_subplot()

for presc, ls in zip(("baruteau", "stahler"), ("-", "--")):
    rows = {
        f"{STALLING_TAGS[s]}": select_with_settings(
            RESULTS,
            stalling_separation=s,
            gas_hardening_prescription=presc,
            nsc_imf_bh_method="default",
            mean_harden_energy_delta=0.9,
            delta_energy_strong_mu=0.1

        )
        for s in STALLING_VALUES
    }

    for s_value in STALLING_VALUES:
        row_name = rows[STALLING_TAGS[s_value]][0]
        if row_name == "":
            continue

        res = RESULTS[row_name]

        df = object_array_to_df(res, "blackholes_merged")

        if not len(df):
            continue

        t = df["time_merged"].value_counts().sort_index().cumsum()
        tk = t.index
        tv = t.values

        c = stall_color(s_value, STALLING_VALUES)
        l = ("variable" if s_value == -1 else f"" if s_value == 0 else f"{s_value:g}")

        ax.plot(tk / 1000000, tv, lw=0.7, color=c, label=f"{presc} {l}", ls=ls)

w = select_with_settings(
    RESULTS,
    stalling_separation=0,
    gas_hardening_prescription="analytical",
    nsc_imf_bh_method="default",
    mean_harden_energy_delta=0.9,
    delta_energy_strong_mu=0.1
)

res = RESULTS[w[0]]

df = object_array_to_df(res, "blackholes_merged")

t = df["time_merged"].value_counts().sort_index().cumsum()
tk = t.index
tv = t.values

ppl, = ax.plot(tk / 1000000, tv, lw=1, color="tab:purple", label=f"_nolegend_", ls=":")

ax.set_xticks([0, .10, .20, .30, .40, .50, .60, .70])
ax.set_xlabel("time [Myr]", fontsize=7)
ax.set_ylabel("cumulative mergers\nper galaxy", fontsize=7)

second_legend = ax.legend(handles=[ppl], labels=["analytical"], fontsize=4.5, frameon=False, title_fontsize=4.5, bbox_to_anchor=(0.4, 0.7))

ax.legend(fontsize=4.5, frameon=False, title=r"stall [$r_g$]", title_fontsize=4.5, ncol=2)

ax.add_artist(second_legend)

fig.suptitle("Cumulative Mergers vs Time", fontsize=7)

In [ ]:
def stall_color(stall, stalling_values):
    if stall == -1.0:
        return "black"

    exclude_var = dict([(s, i) for i, s in enumerate(stalling_values) if s >= 0])
    index = exclude_var[stall]

    return plt.cm.tab10(index)

fig = plt.figure(figsize=(W, H * 1.2))
ax = fig.add_subplot()

for presc, ls in zip(("baruteau", "stahler", "analytical"), ("-", "--", ":")):
    rows = {
        f"{CIRC_HARDENING_TAGS[s]}": select_with_settings(
            RESULTS,
            stalling_separation=0 if presc == "analytical" else 1,
            mean_harden_energy_delta=s,
            gas_hardening_prescription=presc,
            nsc_imf_bh_method="default",
            delta_energy_strong_mu=0.1
        )
        for s in CIRC_HARDENING_MU
    }

    for s_value in CIRC_HARDENING_MU:
        row_name = rows[CIRC_HARDENING_TAGS[s_value]][0]
        if row_name == "":
            continue

        res = RESULTS[row_name]

        df = object_array_to_df(res, "blackholes_merged")

        if not len(df):
            continue

        t = df["time_merged"].value_counts().sort_index().cumsum()
        tk = t.index
        tv = t.values

        c = stall_color(s_value, CIRC_HARDENING_MU)
        l = ("variable" if s_value == -1 else f"" if s_value == 0 else f"{s_value:g}")

        ax.plot(tk / 1000000, tv, lw=0.7, color=c, label=f"{presc} {l}", ls=ls)

ax.set_xticks([0, .10, .20, .30, .40, .50, .60, .70])
ax.set_xlabel("time [Myr]", fontsize=7)
ax.set_ylabel("cumulative mergers\nper galaxy", fontsize=7)

ax.legend(fontsize=4.5, frameon=False, title=r"Circ Encounter $\Delta E$", title_fontsize=4.5, ncol=3)
#ax.set_yscale("log")

fig.suptitle("Cumulative Mergers vs Time", fontsize=7)

#stall sep =1

In [ ]:
# Max Remnant Mass vs Time

fig = plt.figure(figsize=(W, H * 1.2))
ax = fig.add_subplot()

for presc, ls in zip(("baruteau", "stahler"), ("-", "--")):
    rows = {
        f"{STALLING_TAGS[s]}": select_with_settings(
            RESULTS,
            stalling_separation=s,
            gas_hardening_prescription=presc,
            nsc_imf_bh_method="default",
            mean_harden_energy_delta=0.9,
            delta_energy_strong_mu=0.1

        )
        for s in STALLING_VALUES
    }

    for s_value in STALLING_VALUES:
        row_name = rows[STALLING_TAGS[s_value]][0]

        res = RESULTS[row_name]

        df = object_array_to_df(res, "blackholes_merged")
        mass = df.groupby("time_merged")["mass"].max()

        tk = mass.index
        tv = mass.values

        tt = np.zeros(len(tv))

        lowest = tv[0]

        for i, t_val in enumerate(tv):
            if t_val < lowest:
                tt[i] = lowest
            else:
                tt[i] = t_val
                lowest = t_val

        c = stall_color(s_value, STALLING_VALUES)
        l = ("variable" if s_value == -1 else f"" if s_value == 0 else f"{s_value:g}")

        ax.step(tk / 1000000, tt, lw=1, color=c, label=f"{presc} {l}", ls=ls, where="post")

w = select_with_settings(
    RESULTS,
    stalling_separation=0,
    gas_hardening_prescription="analytical",
    nsc_imf_bh_method="default",
    mean_harden_energy_delta=0.9,
    delta_energy_strong_mu=0.1
)

res = RESULTS[w[0]]

df = object_array_to_df(res, "blackholes_merged")
mass = df.groupby("time_merged")["mass"].max()

tk = mass.index
tv = mass.values

tt = np.zeros(len(tv))

lowest = tv[0]

for i, t_val in enumerate(tv):
    if t_val < lowest:
        tt[i] = lowest
    else:
        tt[i] = t_val
        lowest = t_val

ppl, = ax.step(tk / 1000000, tt, lw=1, color="tab:purple", label=f"_nolegend_", ls=":", where="post")

ax.set_xticks([0, .10, .20, .30, .40, .50, .60, .70])
ax.set_xlabel("time [Myr]", fontsize=7)
ax.set_ylabel("remnant mass [M_sun]", fontsize=7)

second_legend = ax.legend(handles=[ppl], labels=["analytical"], fontsize=4.5, frameon=False, title_fontsize=4.5, bbox_to_anchor=(0.4, 0.7))

ax.legend(fontsize=4.5, frameon=False, title=r"stall [$r_g$]", title_fontsize=4.5, ncol=2)

ax.add_artist(second_legend)

fig.suptitle("Max Remnant Mass vs Time", fontsize=7)